# z018 - Modelo común en varios cortes temporales

Modelo común: **solo BAJA+2**, parámetros del trial 34 de z005 (538 árboles), lags 1 y 2, 5 semillas.
Objetivo: ver cuánto varía la ganancia y el mejor N según el mes de train y el de validación.

| Esquema | Train | Valid | Distancia |
|---|---|---|---|
| 1 | 202103 | 202105 | 2 meses |
| 2 | 202103 | 202106 | 3 meses |
| 3 | 202104 | 202106 | 2 meses (el esquema habitual) |
| 4 | 202103 + 202104 | 202105 | 1 mes desde abril |
| 5 | 202103 + 202104 | 202106 | 2 meses desde abril |

Ojo: 202103 no tiene lags (NaN). En el esquema 4 el target de 202104 usa información de 202106 (futuro de mayo):
la ganancia puede estar algo inflada. Las ganancias de mayo y junio no se comparan entre sí (distinta cantidad de BAJA+2:
870 en mayo, 1098 en junio).

In [1]:
# --- 1. Librerías y parámetros comunes ---
import numpy as np
import pandas as pd
import lightgbm as lgb

GANANCIA_ACIERTO = 1_072_500
COSTO_ESTIMULO   = 27_500
MES_VALID = 202106
SEMILLAS  = [501419, 623353, 786349, 834569, 946331]
N_PARTICIONES = 50        # Shuffle Split del test: particiones al azar del 50% de 202106

# Trial 34 de z005 (el mejor en promedio de 5 semillas)
params = {
    "objective": "binary", "learning_rate": 0.005, "min_data_in_leaf": 0,
    "feature_fraction": 0.5, "feature_fraction_bynode": 0.2, "max_bin": 31, "verbose": -1,
    "num_leaves": 137, "min_sum_hessian_in_leaf": 9.755574,
}
ARBOLES = 538

In [2]:
# --- 2. Cargar datos y lags (igual que z003) ---
df = pd.read_parquet("../competencia_01.parquet")
df["clase_binaria"] = (df["clase_ternaria"] == "BAJA+2").astype(int)

no_usar = ["foto_mes", "clase_ternaria", "clase_binaria", "clase_binaria2"]
cols = [c for c in df.columns if c not in no_usar and c != "numero_de_cliente"]

df = df.sort_values(["numero_de_cliente", "foto_mes"]).reset_index(drop=True)
grupo = df.groupby("numero_de_cliente")[cols]
nuevas = []
for k in [1, 2]:
    lag = grupo.shift(k).astype("float32")
    delta = (df[cols] - lag).astype("float32")
    lag.columns   = [f"{c}_lag{k}" for c in cols]
    delta.columns = [f"{c}_delta{k}" for c in cols]
    nuevas += [lag, delta]
df = pd.concat([df] + nuevas, axis=1)
del nuevas, lag, delta, grupo

variables = [c for c in df.columns if c not in no_usar]
valid = df[df["foto_mes"] == MES_VALID]
# La ganancia SIEMPRE se calcula con BAJA+2 (aunque se entrene con otro target)
ganancia_cliente = np.where(valid["clase_ternaria"].values == "BAJA+2", GANANCIA_ACIERTO, -COSTO_ESTIMULO)
print("Variables:", len(variables))

Variables: 761


In [3]:
PREFIJO = "z018 modelo comun"
esquemas = {
    "1: 202103 -> 202105":        ([202103], 202105),
    "2: 202103 -> 202106":        ([202103], 202106),
    "3: 202104 -> 202106":        ([202104], 202106),
    "4: 202103+202104 -> 202105": ([202103, 202104], 202105),
    "5: 202103+202104 -> 202106": ([202103, 202104], 202106),
}
versiones = {"Solo BAJA+2": "clase_binaria"}

# --- 3. Entrenar y evaluar en cada esquema ---
df["clase_binaria2"] = df["clase_ternaria"].isin(["BAJA+1", "BAJA+2"]).astype(int)
filas = []
for esquema, (meses, mes_valid) in esquemas.items():
    train = df[df["foto_mes"].isin(meses)]
    valid = df[df["foto_mes"] == mes_valid]
    g_cliente = np.where(valid["clase_ternaria"].values == "BAJA+2", GANANCIA_ACIERTO, -COSTO_ESTIMULO)
    for version, target in versiones.items():
        datos = lgb.Dataset(train[variables], label=train[target], free_raw_data=False)
        for s in SEMILLAS:
            prob = lgb.train({**params, "seed": s}, datos, num_boost_round=ARBOLES).predict(valid[variables])
            acum = np.cumsum(g_cliente[np.argsort(prob)[::-1]])
            n = int(np.argmax(acum)) + 1
            filas.append({"esquema": esquema, "version": version, "semilla": s,
                          "ganancia_max_M": round(acum[n - 1] / 1e6, 1), "mejor_n": n,
                          "top12000_M": round(acum[11_999] / 1e6, 1)})
        print(esquema, "|", version, "listo", flush=True)
res = pd.DataFrame(filas)
tabla = res.groupby(["esquema", "version"], sort=False)[["ganancia_max_M", "top12000_M", "mejor_n"]].agg(["mean", "std"]).round(1)

1: 202103 -> 202105 | Solo BAJA+2 listo


2: 202103 -> 202106 | Solo BAJA+2 listo


3: 202104 -> 202106 | Solo BAJA+2 listo


4: 202103+202104 -> 202105 | Solo BAJA+2 listo


5: 202103+202104 -> 202106 | Solo BAJA+2 listo


In [4]:
# --- 4. Resultados por esquema ---
tabla

ganancia_max_M      top12000_M        \
                                                 mean  std       mean   std   
esquema                    version                                            
1: 202103 -> 202105        Solo BAJA+2          252.6  1.7      229.7   2.8   
2: 202103 -> 202106        Solo BAJA+2          396.3  4.6      373.6  11.3   
3: 202104 -> 202106        Solo BAJA+2          399.6  2.3      392.9   3.7   
4: 202103+202104 -> 202105 Solo BAJA+2          272.9  1.6      246.8   2.0   
5: 202103+202104 -> 202106 Solo BAJA+2          405.8  4.6      397.1   2.8   

                                        mejor_n          
                                           mean     std  
esquema                    version                       
1: 202103 -> 202105        Solo BAJA+2   7861.4  1122.2  
2: 202103 -> 202106        Solo BAJA+2  15006.2   678.2  
3: 202104 -> 202106        Solo BAJA+2  13212.0  1072.1  
4: 202103+202104 -> 202105 Solo BAJA+2   8677.4   293.0  
5: 202103+202104 -> 202106 Solo BAJA+2  13396.2   367.2

In [5]:
# --- 5. Anotar ---
ruta_res = "resultados_experimentos.csv"
r = pd.read_csv(ruta_res, sep=";", decimal=",", encoding="utf-8-sig")
r = r[~r["descripcion"].str.startswith(PREFIJO)]
for (esquema, version), x in res.groupby(["esquema", "version"], sort=False):
    meses, mes_valid = esquemas[esquema]
    r = pd.concat([r, pd.DataFrame([{
        "fecha": pd.Timestamp.now().strftime("%Y-%m-%d %H:%M"),
        "descripcion": f"{PREFIJO} esquema {esquema} | {version} (prom. 5 semillas)",
        "meses_train": str(meses), "mes_valid": mes_valid, "semilla": str(SEMILLAS), "params": str(params),
        "mejor_n": int(x["mejor_n"].mean()), "ganancia_max_M": round(x["ganancia_max_M"].mean(), 1),
        "ganancia_top12000_M": round(x["top12000_M"].mean(), 1)}])], ignore_index=True)
r["n_enviados"] = r["n_enviados"].astype("Int64")
r.to_csv(ruta_res, sep=";", decimal=",", index=False, encoding="utf-8-sig")
print("Anotado")

Anotado


In [1]:
# --- 6. Comparación normalizada por la cantidad de BAJA+2 del mes de validación ---
# Mayo tiene 870 BAJA+2 y junio 1098: las ganancias en millones no se comparan directo.
import pandas as pd
GANANCIA_ACIERTO = 1_072_500
bajas = pd.read_parquet("../competencia_01.parquet", columns=["foto_mes", "clase_ternaria"])
bajas = bajas[bajas["clase_ternaria"] == "BAJA+2"].groupby("foto_mes").size()

r = pd.read_csv("resultados_experimentos.csv", sep=";", decimal=",", encoding="utf-8-sig")
r = r[r["descripcion"].str.startswith(("z018 modelo comun", "z019 juntar bajas mayo"))
      | r["descripcion"].str.startswith("z009 juntar bajas")].copy()
r["prueba"] = r["descripcion"].str.replace(r" \(prom.*", "", regex=True)
r["BAJA+2 del mes"] = r["mes_valid"].map(bajas)
r["ganancia ideal M"] = (r["BAJA+2 del mes"] * GANANCIA_ACIERTO / 1e6).round(0)
r["% del ideal"] = (r["ganancia_max_M"] / r["ganancia ideal M"] * 100).round(1)
r["llevada a junio M"] = (r["ganancia_max_M"] * bajas[202106] / r["BAJA+2 del mes"]).round(1)
r["N por BAJA+2"] = (r["mejor_n"] / r["BAJA+2 del mes"]).round(1)
r[["prueba", "meses_train", "mes_valid", "BAJA+2 del mes", "ganancia_max_M", "% del ideal",
   "llevada a junio M", "mejor_n", "N por BAJA+2"]].reset_index(drop=True)

,prueba,meses_train,mes_valid,BAJA+2 del mes,ganancia_max_M,% del ideal,llevada a junio M,mejor_n,N por BAJA+2
0,z009 juntar bajas Solo BAJA+2: prom. 5 semilla...,[202104],202106,1098,399.6,33.9,399.6,13212.0,12.0
1,z009 juntar bajas BAJA+1 + BAJA+2: prom. 5 sem...,[202104],202106,1098,421.8,35.8,421.8,14700.0,13.4
2,z018 modelo comun esquema 1: 202103 -> 202105 ...,[202103],202105,870,252.6,27.1,318.8,7861.0,9.0
3,z018 modelo comun esquema 2: 202103 -> 202106 ...,[202103],202106,1098,396.3,33.6,396.3,15006.0,13.7
4,z018 modelo comun esquema 3: 202104 -> 202106 ...,[202104],202106,1098,399.6,33.9,399.6,13212.0,12.0
5,z018 modelo comun esquema 4: 202103+202104 -> ...,"[202103, 202104]",202105,870,272.9,29.2,344.4,8677.0,10.0
6,z018 modelo comun esquema 5: 202103+202104 -> ...,"[202103, 202104]",202106,1098,405.8,34.4,405.8,13396.0,12.2
7,z019 juntar bajas mayo esquema A: 202103 -> 20...,[202103],202105,870,252.6,27.1,318.8,7861.0,9.0
8,z019 juntar bajas mayo esquema A: 202103 -> 20...,[202103],202105,870,274.2,29.4,346.1,7279.0,8.4
9,z019 juntar bajas mayo esquema B: 202104 -> 20...,[202104],202105,870,263.2,28.2,332.2,8549.0,9.8


## Conclusiones (08/10)
| Esquema | Ganancia | % del ideal | Llevada a junio | Mejor N | N por BAJA+2 |
|---|---|---|---|---|---|
| 1: 202103 → 202105 | 252,6 | 27,1% | 318,8 | 7.861 | 9,0 |
| 2: 202103 → 202106 | 396,3 | 33,6% | 396,3 | 15.006 | 13,7 |
| 3: 202104 → 202106 | 399,6 | 33,9% | 399,6 | 13.212 | 12,0 |
| 4: 202103+202104 → 202105 | 272,9 | 29,2% | 344,4 | 8.677 | 10,0 |
| 5: 202103+202104 → 202106 | 405,8 | 34,4% | 405,8 | 13.396 | 12,2 |
- **Mayo es más difícil que junio** aun normalizando (27-29% del ideal vs ~34%): no es solo que tenga menos bajas.
- Sumar marzo al train ayuda un poco (5 vs 3: +6 M; 4 vs 1: +20 M, pero el 4 tiene algo de fuga temporal).
- **El mejor N depende del mes**: ~12-14 clientes por BAJA+2 en junio, ~9-10 en mayo. Con ~1.000 BAJA+2 en agosto,
  12.000 está en el rango de junio; si agosto se pareciera a mayo convendría menos.